<a href="https://colab.research.google.com/github/k2-fsa/colab/blob/master/ctc_decoding_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 介绍

本 colab notebook 详细演示如何用一个 CTC 模型识别一个音频文件。

# 下载模型和测试音频

使用
https://k2-fsa.github.io/sherpa/onnx/pretrained_models/offline-ctc/wenet/index.html#sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10-cantonese

In [1]:
%%shell
wget https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10.tar.bz2
tar xf sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10.tar.bz2
rm sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10.tar.bz2

ls -lh  sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10

--2025-10-11 10:02:38--  https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10.tar.bz2
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/531380835/f0b9b3ee-9ff7-49b9-9fe3-7f1a57bfa568?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-10-11T10%3A57%3A40Z&rscd=attachment%3B+filename%3Dsherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10.tar.bz2&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-10-11T09%3A57%3A10Z&ske=2025-10-11T10%3A57%3A40Z&sks=b&skv=2018-11-09&sig=86oxZqvnGorrgTa8UjCdxE0VMPDWvYTqVl%2B7B52XxCk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZ

In [2]:
%%shell

ls sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/test_wavs

en.wav	    yue-11.wav	yue-14.wav  yue-17.wav	yue-3.wav  yue-6.wav  yue-9.wav
yue-0.wav   yue-12.wav	yue-15.wav  yue-1.wav	yue-4.wav  yue-7.wav  zh.wav
yue-10.wav  yue-13.wav	yue-16.wav  yue-2.wav	yue-5.wav  yue-8.wav


我们要用到其中的
 - model.int8.onnx
 - tokens.txt
 - test_wavs/yue-9.wav

# 安装依赖

In [3]:
%%shell

pip install onnxruntime soundfile kaldi-native-fbank

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.0/322.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.0 MB/s eta 0:00:00


# 加载模型


In [4]:
import onnxruntime as ort
session_opts = ort.SessionOptions()
session_opts.inter_op_num_threads = 1
session_opts.intra_op_num_threads = 1
model = ort.InferenceSession(
    "./sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/model.int8.onnx",
    sess_options=session_opts,
    providers=["CPUExecutionProvider"],
)

# 显示模型的输入和输出


In [5]:
for i in model.get_inputs():
    print(i)

print("-----")

for i in model.get_outputs():
    print(i)

NodeArg(name='x', type='tensor(float)', shape=['N', 'T', 80])
NodeArg(name='x_lens', type='tensor(int64)', shape=['N'])
-----
NodeArg(name='log_probs', type='tensor(float)', shape=['N', 'T', 8629])
NodeArg(name='log_probs_lens', type='tensor(int64)', shape=['N'])


# 读取测试音频

In [6]:
import soundfile as sf
import numpy as np

def load_audio(filename):
    data, sample_rate = sf.read(
        filename,
        always_2d=True,
        dtype="float32",
    )
    assert sample_rate == 16000, sample_rate

    data = data[:, 0]  # use only the first channel
    samples = np.copy(np.ascontiguousarray(data))
    return samples

# 播放测试音频

In [7]:
from IPython.display import Audio, display

display(Audio("sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/test_wavs/yue-9.wav", autoplay=True))

# 提取特征

In [8]:
import kaldi_native_fbank as knf
def create_fbank():
    opts = knf.FbankOptions()
    opts.frame_opts.dither = 0
    opts.mel_opts.num_bins = 80
    opts.frame_opts.snip_edges = False
    opts.mel_opts.debug_mel = False

    fbank = knf.OnlineFbank(opts)
    return fbank


def compute_features(audio) -> np.ndarray:
    fbank = create_fbank()
    assert len(audio.shape) == 1, audio.shape
    # this specific model requires samples in the range [-32768, 32767]
    audio = audio * 32767
    fbank.accept_waveform(16000, audio)
    ans = []
    processed = 0
    while processed < fbank.num_frames_ready:
        ans.append(np.array(fbank.get_frame(processed)))
        processed += 1
    ans = np.stack(ans)
    return ans

# 运行模型

In [9]:
samples = load_audio('sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/test_wavs/yue-9.wav')
features = compute_features(samples)

log_probs,= model.run(
    [
        model.get_outputs()[0].name,
    ],
    {
        model.get_inputs()[0].name: features[None],
        model.get_inputs()[1].name: np.array([features.shape[0]], dtype=np.int64),
    },
)

In [10]:
print(log_probs[0].shape)

(357, 8629)


In [11]:
ids = log_probs[0].argmax(axis=-1).tolist()

In [12]:
print(ids)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2442, 0, 0, 0, 0, 3210, 0, 0, 0, 2141, 0, 0, 0, 0, 8357, 0, 0, 0, 0, 8238, 0, 0, 0, 2005, 0, 0, 4159, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7098, 0, 0, 0, 0, 2363, 0, 0, 0, 0, 0, 2005, 2005, 0, 0, 0, 0, 0, 0, 8610, 0, 0, 0, 0, 0, 4238, 0, 0, 0, 0, 0, 4587, 0, 0, 0, 0, 7683, 0, 0, 0, 2600, 0, 0, 0, 0, 0, 0, 0, 4053, 0, 0, 0, 0, 2453, 0, 0, 0, 0, 0, 2688, 0, 0, 0, 0, 2363, 0, 0, 0, 0, 0, 0, 3220, 0, 0, 0, 0, 7448, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2979, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2442, 0, 0, 0, 0, 0, 3210, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2355, 0, 0, 0, 0, 0, 7550, 0, 0, 0, 0, 2363, 0, 0, 0, 0, 0, 8357, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2135, 0, 0, 0, 0, 0, 8187, 0, 0, 0, 0, 0, 8194, 0, 0, 0, 0, 0, 2008, 0, 0, 0, 0, 2383, 0, 0, 0, 0, 0, 2050, 0, 0, 0, 0, 2502, 0, 0, 0, 2862, 0, 0, 0, 4587, 0, 0, 0, 0, 2453, 0, 0, 0, 0, 2688, 0, 0, 0, 0, 2363, 0, 0, 0, 0, 2862, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# 去重

In [13]:
unique_ids = []
prev = -1
for i in ids:
  if i == prev:
    continue
  prev = i
  unique_ids.append(i)

In [14]:
print(unique_ids)

[0, 2442, 0, 3210, 0, 2141, 0, 8357, 0, 8238, 0, 2005, 0, 4159, 0, 7098, 0, 2363, 0, 2005, 0, 8610, 0, 4238, 0, 4587, 0, 7683, 0, 2600, 0, 4053, 0, 2453, 0, 2688, 0, 2363, 0, 3220, 0, 7448, 0, 2979, 0, 2442, 0, 3210, 0, 2355, 0, 7550, 0, 2363, 0, 8357, 0, 2135, 0, 8187, 0, 8194, 0, 2008, 0, 2383, 0, 2050, 0, 2502, 0, 2862, 0, 4587, 0, 2453, 0, 2688, 0, 2363, 0, 2862, 0, 3513, 0, 4843, 0, 7751, 0, 7891, 0, 7165, 0, 5085, 0, 4017, 0, 5019, 0]


# 去 ε

In [15]:
final_ids = [i for i in unique_ids if i != 0]

In [16]:
print(final_ids)

[2442, 3210, 2141, 8357, 8238, 2005, 4159, 7098, 2363, 2005, 8610, 4238, 4587, 7683, 2600, 4053, 2453, 2688, 2363, 3220, 7448, 2979, 2442, 3210, 2355, 7550, 2363, 8357, 2135, 8187, 8194, 2008, 2383, 2050, 2502, 2862, 4587, 2453, 2688, 2363, 2862, 3513, 4843, 7751, 7891, 7165, 5085, 4017, 5019]


# 查表

In [17]:
def load_tokens():
  id2token = dict()
  with open('sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/tokens.txt') as f:
    for line in f:
      token, idx = line.split()
      id2token[int(idx)] = token
  return id2token

id2token = load_tokens()

In [18]:
tokens = [id2token[i] for i in final_ids]

In [19]:
print(tokens)

['刘', '备', '仲', '马', '鞭', '一', '指', '蜀', '兵', '一', '齐', '掩', '杀', '过', '去', '打', '到', '吴', '兵', '大', '败', '嘿', '刘', '备', '八', '路', '兵', '马', '以', '雷', '霆', '万', '军', '之', '势', '啊', '杀', '到', '吴', '兵', '啊', '尸', '横', '遍', '野', '血', '流', '成', '河']


# 连起来

In [20]:
text = ''.join(tokens)
print(text)

刘备仲马鞭一指蜀兵一齐掩杀过去打到吴兵大败嘿刘备八路兵马以雷霆万军之势啊杀到吴兵啊尸横遍野血流成河


# 再来听听音频

In [21]:
from IPython.display import Audio, display

display(Audio("sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/test_wavs/yue-9.wav", autoplay=True))

# 看下 tokens.txt

In [22]:
! head -n10 sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/tokens.txt

<blank> 0
<unk> 1
' 2
A 3
ABILITY 4
ABLE 5
ABLY 6
AC 7
AD 8
AG 9


In [23]:
! tail -n10 sherpa-onnx-wenetspeech-yue-u2pp-conformer-ctc-zh-en-cantonese-int8-2025-09-10/tokens.txt

龊 8619
龋 8620
龌 8621
龙 8622
龚 8623
龛 8624
龟 8625
龠 8626
龢 8627
<sos/eos> 8628
